> Datacamp Project working with Online Retail data

In [2]:
from pyspark.sql import SparkSession

# create an apache spark session --> SparkSession is an entry point into all functionality in Spark. 
# SparkSession is required if you want to create a dataframe in PySpark...
# the data will be cached on off-heap memory instead of being directly stored on disk and memory amount specified...
spk=SparkSession.builder.appName("datacamp-session").config("spark.memory.offHeap.enabled", "true").config("spark.memory.offHeap.size", "10g").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/14 14:39:38 WARN Utils: Your hostname, anony-threat, resolves to a loopback address: 127.0.1.1; using 192.168.100.52 instead (on interface wlp0s20f3)
25/08/14 14:39:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/14 14:39:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
# Dataframe creation

df=spk.read.csv('online_retail.csv', header=True, escape="/")
df

DataFrame[index: string, InvoiceNo: string, StockCode: string, Description: string, Quantity: string, InvoiceDate: string, UnitPrice: string, CustomerID: string, Country: string]

In [19]:
#display dataframe head
df.show()

+-----+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|index|InvoiceNo|StockCode|         Description|Quantity| InvoiceDate|UnitPrice|CustomerID|       Country|
+-----+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|    0|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/10 8:26|     2.55|     17850|United Kingdom|
|    1|   536365|    71053| WHITE METAL LANTERN|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|    2|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/10 8:26|     2.75|     17850|United Kingdom|
|    3|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|    4|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|    5|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/10 8:26|     7.65|     17850|United Kingdom|
|    6|   536365|    21730|GLASS STAR

In [ ]:
# counting the number of rows in the dataframe
df.count()

100

In [20]:
df.select('CustomerID').distinct().count()
# df.select('COuntry').distinct().count()

7

In [25]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df.groupBy('Country').agg(count_distinct('CustomerID').alias('num_customers')).orderBy(desc('num_customers')).show()

+--------------+-------------+
|       Country|num_customers|
+--------------+-------------+
|United Kingdom|            6|
|        France|            1|
+--------------+-------------+



> Spark SQL

In [3]:
# creating a dataframe

spk_2=SparkSession.builder.appName("sparkSQLSession-1").getOrCreate()

data=[
    (1, 'Mercedes', 14500),
    (2, 'Audi', 10000),
    (3, 'Toyota', 12000),
    (4, 'Nissan', 15000)
]
columns=['id', 'car_model', 'price']

df_2=spk_2.createDataFrame(data,columns)
df_2

25/08/14 14:40:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


DataFrame[id: bigint, car_model: string, price: bigint]

In [4]:
df_2.show()

+---+---------+-----+
| id|car_model|price|
+---+---------+-----+
|  1| Mercedes|14500|
|  2|     Audi|10000|
|  3|   Toyota|12000|
|  4|   Nissan|15000|
+---+---------+-----+



In [5]:
# Registering the dataframe as a temporary view to allow querying with SQL

df_2.createOrReplaceTempView("cars")
spk_2.sql("SELECT * FROM cars").show()

+---+---------+-----+
| id|car_model|price|
+---+---------+-----+
|  1| Mercedes|14500|
|  2|     Audi|10000|
|  3|   Toyota|12000|
|  4|   Nissan|15000|
+---+---------+-----+



In [ ]:
#further querying
# filtering

spk_2.sql("SELECT car_model, price FROM cars WHERE price > 11000").show()

+---------+-----+
|car_model|price|
+---------+-----+
| Mercedes|14500|
|   Toyota|12000|
|   Nissan|15000|
+---------+-----+



In [9]:
# sorting

spk_2.sql("""
SELECT car_model, price 
FROM cars 
ORDER BY price DESC""").show()

+---------+-----+
|car_model|price|
+---------+-----+
|   Nissan|15000|
| Mercedes|14500|
|   Toyota|12000|
|     Audi|10000|
+---------+-----+

